In [ ]:
! pip install pandas pyarrow spacy langdetect pandera tqdm


In [1]:
import os
from pathlib import Path

# Move from /notebooks to project root
os.chdir(Path("..").resolve())

print("New working dir:", os.getcwd())


New working dir: C:\Users\User\Documents\retailmind-prototype


In [2]:
import sys
from pathlib import Path

# Make src/ importable
PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))




In [3]:
from src.ingestion.load_data import load_sgd
from pathlib import Path

SGD_FILE = Path("data/raw/SGD.txt")


In [4]:
df_sgd = load_sgd(SGD_FILE)
len(df_sgd), df_sgd.head(20)


(26666,
     conv_id  turn_id speaker  \
 0         0        1    USER   
 1         0        2  SYSTEM   
 2         0        3    USER   
 3         0        4  SYSTEM   
 4         0        5    USER   
 5         0        6  SYSTEM   
 6         0        7    USER   
 7         0        8  SYSTEM   
 8         0        9    USER   
 9         0       10  SYSTEM   
 10        0       11    USER   
 11        0       12  SYSTEM   
 12        0       13    USER   
 13        0       14  SYSTEM   
 14        0       15    USER   
 15        0       16  SYSTEM   
 16        0       17    USER   
 17        0       18  SYSTEM   
 18        0       19    USER   
 19        1        1    USER   
 
                                                  text     dialog_act  \
 0          What is the weather like on the March 4th?         INFORM   
 1                        In which city should I look?        REQUEST   
 2                         The weather in Mill Valley.         INFORM   
 3   

In [ ]:
df_sgd.tail(15)

In [ ]:
df_sgd.info() 

In [12]:
# 1) make sure types are consistent
df_sgd["conv_id"]   = df_sgd["conv_id"].astype("int64")
df_sgd["turn_id"]   = df_sgd["turn_id"].astype("int64")
df_sgd["speaker"]   = df_sgd["speaker"].astype("category")
df_sgd["dialog_act"] = df_sgd["dialog_act"].astype("category")
df_sgd["is_overall"] = df_sgd["is_overall"].astype("bool")

# 2) create a scalar satisfaction for convenience (e.g. average of scores)
def avg_score(score_list):
    if not score_list:
        return None
    return sum(score_list) / len(score_list)

df_sgd["satisfaction_mean"] = df_sgd["scores"].apply(avg_score)


In [5]:
from src.ingestion.load_data import quick_stats
quick_stats(df_sgd)

Num rows: 26666
Num conversations: 1000
Num OVERALL lines: 1000
Speakers: {'USER': 13833, 'SYSTEM': 12833}


In [ ]:
# Load REDIAL dataset
from src.ingestion.load_redial import load_redial
from pathlib import Path

REDIAL_FILE = Path("data/raw/ReDial.txt")
df_redial = load_redial(REDIAL_FILE)

# Calculate satisfaction_mean
def avg_score(score_list):
    if not score_list:
        return None
    return sum(score_list) / len(score_list)

df_redial["satisfaction_mean"] = df_redial["scores"].apply(avg_score)

# Add label column if dialog_act exists
if "dialog_act" in df_redial.columns:
    df_redial["label"] = df_redial["dialog_act"]


In [ ]:
# REDIAL Statistics
import pandas as pd
import numpy as np

print("=" * 50)
print("REDIAL Dataset Statistics")
print("=" * 50)
print(f"Num rows: {len(df_redial)}")
print(f"Num conversations: {df_redial['conv_id'].nunique()}")
print(f"Num OVERALL lines: {df_redial['is_overall'].sum()}")
print(f"\nSpeakers value counts:")
print(df_redial['speaker'].value_counts())
if 'label' in df_redial.columns:
    print(f"\nLabel value counts:")
    print(df_redial['label'].value_counts())
else:
    print("\nLabel column does not exist")
satisfaction_mean = df_redial['satisfaction_mean'].mean()
if pd.notna(satisfaction_mean):
    print(f"\nSatisfaction mean: {satisfaction_mean:.4f}")
else:
    print("\nSatisfaction mean: N/A")
print("\n" + "=" * 50)
print("Preview head():")
print("=" * 50)
df_redial.head()


In [ ]:
# Load REDIAL-ACTION dataset
from src.ingestion.load_redial_action import load_redial_action
from pathlib import Path

REDIAL_ACTION_FILE = Path("data/raw/ReDial-action.txt")
df_redial_action = load_redial_action(REDIAL_ACTION_FILE)

# Calculate satisfaction_mean (will be None since no scores)
df_redial_action["satisfaction_mean"] = None

# Add label column if dialog_act exists
if "dialog_act" in df_redial_action.columns:
    df_redial_action["label"] = df_redial_action["dialog_act"]


In [ ]:
# REDIAL-ACTION Statistics
import pandas as pd
import numpy as np

print("=" * 50)
print("REDIAL-ACTION Dataset Statistics")
print("=" * 50)
print(f"Num rows: {len(df_redial_action)}")
print(f"Num conversations: {df_redial_action['conv_id'].nunique()}")
print(f"Num OVERALL lines: {df_redial_action['is_overall'].sum()}")
print(f"\nSpeakers value counts:")
print(df_redial_action['speaker'].value_counts())
if 'label' in df_redial_action.columns:
    print(f"\nLabel value counts:")
    print(df_redial_action['label'].value_counts())
else:
    print("\nLabel column does not exist")
satisfaction_mean = df_redial_action['satisfaction_mean'].mean()
if pd.notna(satisfaction_mean):
    print(f"\nSatisfaction mean: {satisfaction_mean:.4f}")
else:
    print("\nSatisfaction mean: N/A (no scores available)")
print("\n" + "=" * 50)
print("Preview head():")
print("=" * 50)
df_redial_action.head()


In [ ]:
# Load MWOZ dataset
from src.ingestion.load_mwoz import load_mwoz
from pathlib import Path

MWOZ_FILE = Path("data/raw/MWOZ.txt")
df_mwoz = load_mwoz(MWOZ_FILE)

# Calculate satisfaction_mean
def avg_score(score_list):
    if not score_list:
        return None
    return sum(score_list) / len(score_list)

df_mwoz["satisfaction_mean"] = df_mwoz["scores"].apply(avg_score)

# Add label column if dialog_act exists
if "dialog_act" in df_mwoz.columns:
    df_mwoz["label"] = df_mwoz["dialog_act"]


In [ ]:
# MWOZ Statistics
import pandas as pd
import numpy as np

print("=" * 50)
print("MWOZ Dataset Statistics")
print("=" * 50)
print(f"Num rows: {len(df_mwoz)}")
print(f"Num conversations: {df_mwoz['conv_id'].nunique()}")
print(f"Num OVERALL lines: {df_mwoz['is_overall'].sum()}")
print(f"\nSpeakers value counts:")
print(df_mwoz['speaker'].value_counts())
if 'label' in df_mwoz.columns:
    print(f"\nLabel value counts:")
    print(df_mwoz['label'].value_counts())
else:
    print("\nLabel column does not exist")
satisfaction_mean = df_mwoz['satisfaction_mean'].mean()
if pd.notna(satisfaction_mean):
    print(f"\nSatisfaction mean: {satisfaction_mean:.4f}")
else:
    print("\nSatisfaction mean: N/A")
print("\n" + "=" * 50)
print("Preview head():")
print("=" * 50)
df_mwoz.head()
